# Lab 6 - Plotting Hamming Weight

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous part of SCA101, we saw that a component of these power traces that we've been capturing is based on the Hamming weight of data manipulated by the target. By getting our target to manipulate Hamming weights with large differences, we were able to detect this difference.*

*In this lab, we'll look at showcasing not just that this difference exists and is measurable, but that it's also linear.*

**LEARNING OUTCOMES:**

* Understand Hamming weight
* Calculate Hamming weight of a byte
* Using Numpy to group traces by Hamming weight
* Plotting power consumption vs. Hamming weight of internal values

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ [Lab 5 - Large Hamming Weight Swings](../Lab%205%20-%20Large%20Hamming%20Weight%20Swings/Lab.ipynb).

## Intro

In the previous lab, we tested the idea that there's some relationship between power consumption and the Hamming weight of data being manipulated by our target. When averaging a large amount of traces grouped into high Hamming weights and low Hamming weights we found a distinct difference between our two groups. This lab also served as an introduction to using numpy to perform operations across entire arrays at once, which we'll be using going forward in this course.

In this lab, we're going to confirm the linear relationship between the Hamming weight of this data and the power consumption at the point we looked at in the last lab.

## Setup

Setup is the same as in previous labs:

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C SS_VER=SS_VER_1_1 -j

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

## Capturing Traces

In the last lab, we restricted the first byte of our plaintext to either being the maximum or minimum Hamming weight, 0xFF and 0x00, respectively. In this lab, we need the full range of Hamming weights. This means that we don't need
to modify our plaintext; however, we will need to capture many more traces. This is because there are many more numbers with a middle Hamming weight than a large or small: an 8-bit number has 70 possible values with a Hamming weight of 4, while there's only
1 with Hamming weights of 0 or 8.

Of course, you could try capturing an even distrubtion of Hamming weights, but doing a simple random capture like this is much easier.

**Create your capture loop below. Capture 5000 traces:**

Again, we'll do a quick sanity plot:

In [ ]:
cw.plot(project.traces[0]) * cw.plot(project.traces[1])

## Hamming Weight

Recall that the reason that there's a relationship between power consumption and the microcontroller's internal data is that setting this data takes power. We've also seen that the more data that is set, the greater the average power draw. It's not far fetched, then, that there should be some sort of consistant relationship between the number of bits set to 1, called the **Hamming weight** and the power consumed by doing so.

Hamming weight, despite being a pretty simple idea, actually isn't trivial to calculate (see https://en.wikipedia.org/wiki/Hamming_weight#Efficient_implementation). You can write a function to do this, but in Python it's far easier to just convert to a string of bits and count the `"1"`s, then use that to create a lookup table.

**Do that below**

We can confirm what I said earlier about the distrubtion of Hamming weights for 8-bit numbers using the unique function from numpy:

In [ ]:
u, c = np.unique(HW, return_counts=True)
dict(zip(u, c))

## Grouping Our Traces

Next, we need to group our power traces by the Hamming weight of their corresponding plaintext. Luckily, this is very similar to what we did in the previous lab.

As a reminder, here's what the code from the last lab looked like:

```python
is_zero = project.plaintexts[:,0] == 0x00
nzeros_list = project.traces[is_zero,:]
nones_list = project.traces[~is_zero,:]
print((nzeros_list == zeros_list).all())
```

You need to do the following:

1. Change the first line to reference the Hamming weight of the plaintext (`HW[project.plaintexts[:,0]]`)
2. Change the conditional to reference a given Hamming weight instead of 0x00
3. Delete the inverse conditional line
4. Loop through the Hamming weights and add the traces to a list

At the end you should have 9 lists of traces.

Now, we want to plot the Hamming weight at the point where our data is being manipulated. Luckily, we actually found this in the previous lab when we plotted the difference
between 0xFF and 0x00. If you've forgotten your point of interest, you can quickly plot the difference between your minimum and maximum Hamming weight groups again. You should have
enough traces in these groups for a reasonable plot.

**Now you'll need to loop through the Hamming weight groups and take the mean of each group at the spike in the plot above:**

If you've done everything corretly you should have a straight line.

## Correlation

Our plot here looks very linear, but is there an objective way to measure linearity? Indeed there is, using the Pearson Correlation Coefficient. We won't go into the details here, but in short, this correlation varies from -1 to 0 to 1, with -1 and 1 being perfect linear correlations and 0 being no linear correlation.

Numpy has a correlation function called `corrcoef` which you can use to calculate the correlation using a list of the Hamming weights and the averages that we collected. You should get a 2x2 matrix back, with the [1,0] and/or [0,1] positions representing the correlation that we're interested in. You should see a very high correlation.

## Plotting the Groups

One more thing we can do is to plot the averages of entire Hamming weight trace groups. If you zoom in where our plaintext is being loaded (aka around our point of interest), you should see a nice colourful distribution of lines corresponding to our different Hamming weight groups.

## Conclusions & Next Steps

With this lab, you should be reasonably convinced that there is a linear relationship between the Hamming weight of data being set in a microcontroller and the power it consumes from doing so.

In the next lab, we'll see that this linear relationship can be used to recover AES keys using ChipWhisperer Analyzer.

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>